# HMM Profile Drafter Test

This notebook contains only the code needed to run the PF00535 HMM profile drafter benchmark for ProstT5 inverse folding.

Run the cells top to bottom. The benchmark itself only executes in the last cell.

## 1. Mount Drive and expose the project files

This cell connects the Colab runtime to Google Drive and sets `DRIVE_ROOT` to the notebook folder that contains the HMM benchmark assets.

It also appends that folder to `sys.path`, which makes local modules such as `hmm_drafter.py` importable from later cells.

If the expected Drive folder is missing, the cell fails early so the rest of the notebook does not run with a broken path configuration.

In [1]:
from pathlib import Path
import sys, shutil

DRIVE_ROOT = Path('/content')
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

# Notebook expects PF00535.hmm under hmm_data/, you uploaded it flat into /content/
(DRIVE_ROOT / 'hmm_data').mkdir(exist_ok=True)
src = DRIVE_ROOT / 'PF00535.hmm'
dst = DRIVE_ROOT / 'hmm_data' / 'PF00535.hmm'
if src.exists() and not dst.exists():
    shutil.move(str(src), str(dst))

print('DRIVE_ROOT =', DRIVE_ROOT)
print('files:', sorted(p.name for p in DRIVE_ROOT.iterdir() if p.is_file()))
print('hmm_data:', [p.name for p in (DRIVE_ROOT/'hmm_data').iterdir()])

DRIVE_ROOT = /content
files: ['build_hmm.py', 'hmm_drafter.py']
hmm_data: ['PF00535.hmm']


## 2. Install the runtime dependencies

This cell installs the libraries that are required inside the Colab environment.

- `pyhmmer` is used for HMM profile handling.
- `torch` provides model execution.
- `pandas`, `accelerate`, `transformers`, `protobuf`, and `sentencepiece` support ProstT5 loading, tokenization, and result aggregation.

These installs are notebook-local setup steps; they do not define any benchmark logic yet.

In [2]:
%pip install pyhmmer==0.12.0
%pip install "torch>=2.0"
%pip install -q pandas 'accelerate>=0.26.0' 'transformers==4.46.3' 'protobuf>=5.27' sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.7 MB/s eta 0:00:00


## 3. Import modules and define benchmark configuration

This section prepares the runtime used by the rest of the notebook.

It imports standard utilities, loads the speculative decoding helpers from `hmm_drafter.py`, and defines the benchmark constants: the Pfam family, drafter variants, `K` values, dataset size targets, cache locations, and download policy.

At the end, it resolves the compute device and datatype so later cells can run on GPU when available and fall back to CPU otherwise.

In [3]:
import gc
import os
import platform
import shutil
import stat
import subprocess
import sys
import tarfile
import time
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, T5Tokenizer

NOTEBOOK_DIR = DRIVE_ROOT
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hmm_drafter import (
    HMMDrafter,
    PrefixAwareHMMDrafter,
    encdec_greedy_reference,
    spec_decode_greedy,
)

PROSTT5_NAME = 'Rostlab/ProstT5_fp16'
PFAM_ID = 'PF00535'
USE_FP16 = True
DRAFTER_VARIANTS = ['naive', 'prefix_aware']
HMM_K_VALUES = [1, 2, 4, 8, 16]
HMM_REPEATS = 1
HMM_WARMUP = 0
HMM_MIN_CASE_LENGTH = 80
HMM_MAX_CASE_LENGTH = 450
HMM_IN_FAMILY_TARGET = 3
HMM_OUT_FAMILY_TARGET = 3
HMM_QUERY_OVERFETCH = 150
ALLOW_NETWORK_DOWNLOADS = True

HMM_SEED_CASES = {
    'in_family': [
        'P39621',
        'Q9Y263',
        'P01112',
    ],
    'out_of_family': [
        'P04637',
        'A0A6G0XC32',
        'P01308',
    ],
}

DRAFTER_MAP = {
    'naive': HMMDrafter,
    'prefix_aware': PrefixAwareHMMDrafter,
}

DATA_DIR = NOTEBOOK_DIR / 'benchmark_data'
RESULTS_DIR = NOTEBOOK_DIR / 'benchmark_results'
HMM_DATA_DIR = NOTEBOOK_DIR / 'hmm_data'
HMM_HMM_PATH = HMM_DATA_DIR / f'{PFAM_ID}.hmm'
FOLDSEEK_DIR = NOTEBOOK_DIR / 'foldseek_bin'
LOCAL_FOLDSEEK_BIN = Path('/tmp/foldseek_bin')
TEST_AA_FASTA = DATA_DIR / 'test_set_AA.fasta'
TEST_3DI_FASTA = DATA_DIR / 'test_set_3Di.fasta'
DISPLAY_FLOAT_FORMAT = '{:.4f}'.format

DATA_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
HMM_DATA_DIR.mkdir(exist_ok=True)

print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('HMM_HMM_PATH =', HMM_HMM_PATH)
print('build_hmm.py exists:', (NOTEBOOK_DIR / 'build_hmm.py').exists())
print('hmm_drafter.py exists:', (NOTEBOOK_DIR / 'hmm_drafter.py').exists())
print('PFAM HMM exists:', HMM_HMM_PATH.exists())
print('Cached AA FASTA exists:', TEST_AA_FASTA.exists())
print('Cached 3Di FASTA exists:', TEST_3DI_FASTA.exists())
print('ALLOW_NETWORK_DOWNLOADS =', ALLOW_NETWORK_DOWNLOADS)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if (USE_FP16 and device.type == 'cuda') else torch.float32
print(f'torch={torch.__version__} device={device} dtype={dtype}')

NOTEBOOK_DIR = /content
HMM_HMM_PATH = /content/hmm_data/PF00535.hmm
build_hmm.py exists: True
hmm_drafter.py exists: True
PFAM HMM exists: True
Cached AA FASTA exists: False
Cached 3Di FASTA exists: False
ALLOW_NETWORK_DOWNLOADS = True
torch=2.10.0+cu128 device=cuda:0 dtype=torch.float16


## 4. Helper functions for environment setup, caching, and case assembly

This is the main utility section of the notebook. It contains small helper functions that keep the final benchmark cell readable by separating setup, data loading, and case-selection logic.

Function overview:

- `_sync()`: waits for queued CUDA work to finish so timings are measured accurately.
- `_reset_peak_mem()`: clears PyTorch's tracked peak GPU memory before a measured run.
- `_peak_mem_gb()`: reports the maximum allocated GPU memory in gigabytes for the current run.
- `_prepare_foldseek_binary(path)`: makes sure an existing Foldseek binary is executable and stages it to local storage when needed.
- `_foldseek_url()`: chooses the correct Foldseek download URL for Linux or macOS.
- `_have_foldseek()`: searches common locations for an already available Foldseek binary.
- `_install_foldseek()`: downloads and unpacks Foldseek if it is not already present.
- `_ensure_foldseek()`: returns a usable Foldseek binary, installing it only when necessary.
- `_read_fasta_map(path, lowercase=False)`: reads a FASTA file into a dictionary keyed by record ID.
- `_load_cached_case_from_fasta(uid)`: loads a protein case from the cached amino-acid and 3Di FASTA files.
- `_ensure_ss_header_db(work_dir)`: copies Foldseek sidecar header files so secondary-structure exports can be read correctly.
- `_load_cached_case_from_workdir(uid)`: reconstructs a cached case from a prior Foldseek working directory.
- `_afdb_pdb_url(uniprot_id)`: queries the AlphaFold DB API to find the PDB download URL for one UniProt ID.
- `_download_pdb(uniprot_id, out_dir)`: downloads a structure file unless it is already cached locally.
- `_extract_3di(uniprot_id, pdb_path, foldseek_bin)`: runs Foldseek to produce aligned amino-acid and 3Di sequences from a PDB file.
- `_uniprot_search_tsv(query, size)`: queries UniProt for candidate proteins or falls back to seeded IDs when network access is disabled.
- `_label_for_candidate(uid, bucket, candidate_name_map)`: builds a readable label for result tables.
- `_get_case_record(uid, foldseek_bin)`: loads a case from cache when possible and otherwise downloads and derives it.
- `assemble_hmm_cases()`: builds the final benchmark case list by collecting in-family and out-of-family proteins, validating lengths, and attaching metadata.

Together, these helpers prepare the protein examples and external tools needed before ProstT5 benchmarking begins.

In [4]:
def _sync():
    if device.type == 'cuda':
        torch.cuda.synchronize()


def _reset_peak_mem():
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)


def _peak_mem_gb() -> float:
    if device.type != 'cuda':
        return 0.0
    return torch.cuda.max_memory_allocated(device) / 1e9


def _prepare_foldseek_binary(path: str | Path) -> str:
    candidate = Path(path)
    if not candidate.exists():
        raise FileNotFoundError(candidate)
    if str(candidate).startswith('/content/drive/'):
        LOCAL_FOLDSEEK_BIN.mkdir(parents=True, exist_ok=True)
        staged = LOCAL_FOLDSEEK_BIN / candidate.name
        shutil.copy(candidate, staged)
        staged.chmod(staged.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
        return str(staged)
    if not os.access(candidate, os.X_OK):
        candidate.chmod(candidate.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return str(candidate)


def _foldseek_url() -> str | None:
    if platform.system() == 'Linux':
        return 'https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz'
    if platform.system() == 'Darwin':
        return 'https://mmseqs.com/foldseek/foldseek-osx-universal.tar.gz'
    return None


def _have_foldseek() -> str | None:
    on_path = shutil.which('foldseek')
    if on_path:
        return _prepare_foldseek_binary(on_path)

    candidates = [
        FOLDSEEK_DIR / 'bin' / 'foldseek',
        NOTEBOOK_DIR / 'foldseek' / 'bin' / 'foldseek',
        Path('/content/foldseek/bin/foldseek'),
        Path.cwd() / 'foldseek' / 'bin' / 'foldseek',
        Path('/tmp/foldseek/bin/foldseek'),
        Path('/usr/local/bin/foldseek'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return _prepare_foldseek_binary(candidate)

    for root in {NOTEBOOK_DIR, Path.cwd(), Path('/content'), Path('/tmp')}:
        if not root.exists():
            continue
        for hit in root.glob('**/foldseek/bin/foldseek'):
            if hit.is_file():
                return _prepare_foldseek_binary(hit)
    return None


def _install_foldseek() -> str:
    if not ALLOW_NETWORK_DOWNLOADS:
        raise FileNotFoundError(
            'foldseek binary not found locally. Put it under MyDrive/HMM/foldseek_bin/bin/foldseek '
            'or enable ALLOW_NETWORK_DOWNLOADS.'
        )

    FOLDSEEK_DIR.mkdir(parents=True, exist_ok=True)
    url = _foldseek_url()
    if url is None:
        raise RuntimeError(f'Auto-install not supported on {platform.system()}/{platform.machine()}.')

    tarball = FOLDSEEK_DIR / 'foldseek.tar.gz'
    if not tarball.exists():
        print(f'Downloading Foldseek from {url} ...')
        urllib.request.urlretrieve(url, tarball)

    with tarfile.open(tarball, 'r:gz') as archive:
        archive.extractall(FOLDSEEK_DIR)

    src_bin = FOLDSEEK_DIR / 'foldseek' / 'bin' / 'foldseek'
    dst_bin = FOLDSEEK_DIR / 'bin' / 'foldseek'
    if not src_bin.exists():
        raise RuntimeError(f'foldseek binary not found after extracting {tarball}')

    dst_bin.parent.mkdir(parents=True, exist_ok=True)
    if dst_bin.exists():
        dst_bin.unlink()
    shutil.copy(src_bin, dst_bin)
    dst_bin.chmod(dst_bin.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    return _prepare_foldseek_binary(dst_bin)


def _ensure_foldseek() -> str:
    existing = _have_foldseek()
    return existing if existing is not None else _install_foldseek()


def _read_fasta_map(path: Path, lowercase: bool = False) -> dict[str, str]:
    if not path.exists():
        return {}

    records: dict[str, str] = {}
    current_id = None
    parts: list[str] = []
    for line in path.read_text().splitlines():
        if line.startswith('>'):
            if current_id is not None:
                seq = ''.join(parts)
                records[current_id] = seq.lower() if lowercase else seq
            current_id = line[1:].split()[0]
            parts = []
        else:
            parts.append(line.strip())

    if current_id is not None:
        seq = ''.join(parts)
        records[current_id] = seq.lower() if lowercase else seq
    return records


def _load_cached_case_from_fasta(uid: str) -> dict | None:
    aa_map = _read_fasta_map(TEST_AA_FASTA, lowercase=False)
    di_map = _read_fasta_map(TEST_3DI_FASTA, lowercase=True)
    aa_seq = aa_map.get(uid)
    three_di_seq = di_map.get(uid)
    if not aa_seq or not three_di_seq:
        return None
    if len(aa_seq) != len(three_di_seq):
        raise ValueError(f'{uid}: cached FASTA lengths disagree ({len(aa_seq)} vs {len(three_di_seq)})')
    return {
        'uid': uid,
        'aa': aa_seq,
        '3di': three_di_seq,
        'length': len(three_di_seq),
    }


def _ensure_ss_header_db(work_dir: Path) -> None:
    src_base = work_dir / 'queryDB_h'
    dst_base = work_dir / 'queryDB_ss_h'
    for suffix in ['', '.dbtype', '.index']:
        src = Path(f'{src_base}{suffix}')
        dst = Path(f'{dst_base}{suffix}')
        if src.exists() and not dst.exists():
            shutil.copy(src, dst)


def _load_cached_case_from_workdir(uid: str) -> dict | None:
    work_dir = DATA_DIR / 'foldseek_work' / uid
    aa_db = work_dir / 'queryDB'
    di_db = work_dir / 'queryDB_ss'
    if not (work_dir.exists() and aa_db.exists() and di_db.exists()):
        return None

    foldseek_bin = _have_foldseek()
    if foldseek_bin is None:
        foldseek_bin = _install_foldseek()

    _ensure_ss_header_db(work_dir)

    aa_fasta = work_dir / 'queryDB.fasta'
    di_fasta = work_dir / 'queryDB_ss.fasta'
    if not aa_fasta.exists():
        subprocess.run([foldseek_bin, 'convert2fasta', str(aa_db), str(aa_fasta)], check=True)
    if not di_fasta.exists():
        subprocess.run([foldseek_bin, 'convert2fasta', str(di_db), str(di_fasta)], check=True)

    aa_seq = ''.join(line.strip() for line in aa_fasta.read_text().splitlines() if not line.startswith('>')).replace('-', '')
    three_di_seq = ''.join(line.strip() for line in di_fasta.read_text().splitlines() if not line.startswith('>')).replace('-', '').lower()
    if not aa_seq or not three_di_seq:
        return None
    if len(aa_seq) != len(three_di_seq):
        raise ValueError(f'{uid}: foldseek workdir lengths disagree ({len(aa_seq)} vs {len(three_di_seq)})')
    return {
        'uid': uid,
        'aa': aa_seq,
        '3di': three_di_seq,
        'length': len(three_di_seq),
    }


def _afdb_pdb_url(uniprot_id: str) -> str:
    api = f'https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}'
    with urllib.request.urlopen(api, timeout=15) as response:
        body = __import__('json').loads(response.read())
    if not body or not isinstance(body, list):
        raise RuntimeError(f'AFDB API returned no entries for {uniprot_id}')
    pdb_url = body[0].get('pdbUrl')
    if not pdb_url:
        raise RuntimeError(f'AFDB API entry for {uniprot_id} has no pdbUrl')
    return pdb_url


def _download_pdb(uniprot_id: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    pdb_path = out_dir / f'AF-{uniprot_id}-F1-model.pdb'
    if pdb_path.exists() and pdb_path.stat().st_size > 0:
        return pdb_path
    if not ALLOW_NETWORK_DOWNLOADS:
        raise FileNotFoundError(
            f'Cached PDB not found for {uniprot_id}: {pdb_path}. '
            'Add the file under MyDrive/HMM/benchmark_data/afdb_pdbs or enable ALLOW_NETWORK_DOWNLOADS.'
        )
    print(f'Downloading PDB for {uniprot_id} ...')
    urllib.request.urlretrieve(_afdb_pdb_url(uniprot_id), pdb_path)
    return pdb_path


def _extract_3di(uniprot_id: str, pdb_path: Path, foldseek_bin: str) -> tuple[str, str]:
    work_dir = DATA_DIR / 'foldseek_work' / uniprot_id
    work_dir.mkdir(parents=True, exist_ok=True)
    pdb_dir = work_dir / 'pdbs'
    pdb_dir.mkdir(exist_ok=True)

    target_path = pdb_dir / pdb_path.name
    if not target_path.exists():
        shutil.copy(pdb_path, target_path)

    db_path = work_dir / 'queryDB'
    subprocess.run([foldseek_bin, 'createdb', str(pdb_dir), str(db_path)], check=True)
    _ensure_ss_header_db(work_dir)

    aa_fasta = work_dir / 'queryDB.fasta'
    di_fasta = work_dir / 'queryDB_ss.fasta'
    subprocess.run([foldseek_bin, 'convert2fasta', str(db_path), str(aa_fasta)], check=True)
    subprocess.run([foldseek_bin, 'convert2fasta', str(db_path) + '_ss', str(di_fasta)], check=True)

    def _read_one(path: Path, lowercase: bool) -> str:
        chunks = []
        for line in path.read_text().splitlines():
            if line.startswith('>'):
                continue
            chunks.append(line.strip())
        text = ''.join(chunks).replace('-', '')
        return text.lower() if lowercase else text

    return _read_one(aa_fasta, lowercase=False), _read_one(di_fasta, lowercase=True)


def _uniprot_search_tsv(query: str, size: int) -> list[dict]:
    if not ALLOW_NETWORK_DOWNLOADS:
        seeded = []
        seen = set()
        for bucket_ids in HMM_SEED_CASES.values():
            for uid in bucket_ids:
                if uid in seen:
                    continue
                seen.add(uid)
                seeded.append({
                    'uid': uid,
                    'protein_name': uid,
                    'organism_name': 'cached',
                    'length': 0,
                })
        return seeded

    params = urllib.parse.urlencode({
        'query': query,
        'fields': 'accession,protein_name,organism_name,length',
        'format': 'tsv',
        'size': size,
    })
    url = f'https://rest.uniprot.org/uniprotkb/search?{params}'
    with urllib.request.urlopen(url, timeout=60) as response:
        lines = response.read().decode('utf-8').splitlines()

    rows = []
    for line in lines[1:]:
        parts = line.split('\t')
        if len(parts) != 4:
            continue
        accession, protein_name, organism_name, length = parts
        rows.append({
            'uid': accession,
            'protein_name': protein_name.strip() or accession,
            'organism_name': organism_name.strip() or 'unknown organism',
            'length': int(length),
        })
    return rows


def _label_for_candidate(uid: str, bucket: str, candidate_name_map: dict[str, dict]) -> str:
    entry = candidate_name_map.get(uid, {})
    protein_name = entry.get('protein_name', uid).replace(';', ',')[:72]
    return f'{bucket}: {protein_name}'


def _get_case_record(uid: str, foldseek_bin: str) -> dict:
    for loader in (_load_cached_case_from_fasta, _load_cached_case_from_workdir):
        cached = loader(uid)
        if cached is not None:
            return cached

    pdb_path = _download_pdb(uid, DATA_DIR / 'afdb_pdbs')
    aa_seq, three_di_seq = _extract_3di(uid, pdb_path, foldseek_bin)
    if len(aa_seq) != len(three_di_seq):
        raise ValueError(f'{uid}: AA and 3Di lengths disagree ({len(aa_seq)} vs {len(three_di_seq)})')
    return {
        'uid': uid,
        'aa': aa_seq,
        '3di': three_di_seq,
        'length': len(three_di_seq),
    }


def assemble_hmm_cases() -> list[dict]:
    if not HMM_HMM_PATH.exists():
        if not ALLOW_NETWORK_DOWNLOADS:
            raise FileNotFoundError(
                f'PFAM HMM not found at {HMM_HMM_PATH}. Place PF00535.hmm under MyDrive/HMM/hmm_data '
                'or enable ALLOW_NETWORK_DOWNLOADS.'
            )
        subprocess.run([sys.executable, str(NOTEBOOK_DIR / 'build_hmm.py')], cwd=NOTEBOOK_DIR, check=True)

    foldseek_bin = _ensure_foldseek()
    in_family_query = (
        f'reviewed:true AND length:[{HMM_MIN_CASE_LENGTH} TO {HMM_MAX_CASE_LENGTH}] '
        f'AND xref:pfam-{PFAM_ID}'
    )
    out_of_family_query = (
        f'reviewed:true AND length:[{HMM_MIN_CASE_LENGTH} TO {HMM_MAX_CASE_LENGTH}] '
        f'NOT xref:pfam-{PFAM_ID}'
    )

    in_family_candidates = _uniprot_search_tsv(in_family_query, HMM_QUERY_OVERFETCH)
    out_of_family_candidates = _uniprot_search_tsv(out_of_family_query, HMM_QUERY_OVERFETCH)
    candidate_name_map = {entry['uid']: entry for entry in [*in_family_candidates, *out_of_family_candidates]}

    def _assemble_bucket(bucket: str, seeds: list[str], candidates: list[dict], target_count: int) -> list[dict]:
        selected = []
        seen = set()
        for uid in [*seeds, *[entry['uid'] for entry in candidates]]:
            if uid in seen:
                continue
            seen.add(uid)
            try:
                rec = _get_case_record(uid, foldseek_bin)
            except Exception as exc:
                print(f'SKIP {uid} ({bucket}): {exc}')
                continue
            if rec['length'] < HMM_MIN_CASE_LENGTH or rec['length'] > HMM_MAX_CASE_LENGTH:
                continue
            rec['bucket'] = bucket
            rec['label'] = _label_for_candidate(uid, bucket, candidate_name_map)
            selected.append(rec)
            if len(selected) >= target_count:
                break
        if not selected:
            raise RuntimeError(f'No usable cases found for {bucket}.')
        if len(selected) < target_count:
            print(f'WARNING: only found {len(selected)} cases for {bucket} (target={target_count})')
        return selected

    hmm_cases = []
    hmm_cases.extend(_assemble_bucket('in_family', HMM_SEED_CASES['in_family'], in_family_candidates, HMM_IN_FAMILY_TARGET))
    hmm_cases.extend(_assemble_bucket('out_of_family', HMM_SEED_CASES['out_of_family'], out_of_family_candidates, HMM_OUT_FAMILY_TARGET))
    return hmm_cases

## 5. Load the ProstT5 tokenizer and model

This cell downloads or restores the ProstT5 tokenizer and sequence-to-sequence model, moves the model to the chosen device, and switches it to evaluation mode.

If half precision is enabled on CUDA, the model is explicitly converted to `float16` so the benchmark uses the intended inference configuration.

In [5]:
tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    PROSTT5_NAME,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
).to(device).eval()

# Upcast on GPU for bit-exact verification (T4 has 16 GB; fp32 model ~11 GB)
model = model.float()
torch.cuda.empty_cache()

print(f'ProstT5 dtype = {next(model.parameters()).dtype}')
print(f'allocated     = {torch.cuda.memory_allocated()/1024**3:.2f} GB')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

ProstT5 dtype = torch.float32
allocated     = 10.50 GB


## 6. Define the per-run benchmark routine

This section defines `time_hmm_spec(...)`, the function that executes one benchmark configuration.

`time_hmm_spec` does four things:

- optionally performs warm-up runs,
- runs speculative decoding with one drafter class and one `K` value,
- measures latency, throughput, acceptance statistics, and peak GPU memory,
- decodes the generated amino-acid prediction and returns both the metrics rows and the last predicted sequence.

The `@torch.inference_mode()` decorator disables gradient tracking, which reduces overhead and matches inference-only usage.

In [6]:
@torch.inference_mode()
def time_hmm_spec(uid: str, rec: dict, drafter_cls, k: int, ref_ids: list[int]) -> tuple[list[dict], str]:
    variant = 'prefix_aware' if drafter_cls is PrefixAwareHMMDrafter else 'naive'
    rows = []
    last_pred = ''

    for _ in range(HMM_WARMUP):
        warm_drafter = drafter_cls(HMM_HMM_PATH, rec['aa'], tokenizer)
        spec_decode_greedy(model, tokenizer, rec['3di'], warm_drafter, K=k, device=device)
    _sync()

    for repeat in range(HMM_REPEATS):
        drafter = drafter_cls(HMM_HMM_PATH, rec['aa'], tokenizer)
        _reset_peak_mem()
        _sync()
        t0 = time.perf_counter()
        spec_ids, stats = spec_decode_greedy(model, tokenizer, rec['3di'], drafter, K=k, device=device)
        _sync()
        dt = time.perf_counter() - t0

        pred_aa = ''.join(tokenizer.decode(spec_ids, skip_special_tokens=True).split())
        last_pred = pred_aa
        accepted = int(stats['accepted'])
        proposed = int(stats['proposed'])
        rows.append({
            'protein_id': uid,
            'bucket': rec['bucket'],
            'label': rec['label'],
            'variant': variant,
            'k': k,
            'length': rec['length'],
            'repeat': repeat,
            'bit_exact': spec_ids == ref_ids,
            'accepted': accepted,
            'proposed': proposed,
            'accept_rate': (accepted / proposed) if proposed else 0.0,
            'steps': int(stats['steps']),
            'extra_tokens': int(stats['extra_tokens']),
            'reanchor_calls': int(getattr(drafter, 'reanchor_calls', 0)),
            'wall_s': dt,
            'tokens_per_s': len(spec_ids) / dt if dt > 0 else float('nan'),
            'peak_vram_gb': _peak_mem_gb(),
            'pred_len': len(pred_aa),
        })
    return rows, last_pred

In [7]:
hmm_cases = assemble_hmm_cases()
print(f'Verifying bit-exact (fp32) on {len(hmm_cases)} cases at K=4, both variants\n')

results = []
for rec in hmm_cases:
    ref_ids = encdec_greedy_reference(model, tokenizer, rec['3di'], device)
    for variant_name, drafter_cls in DRAFTER_MAP.items():
        drafter = drafter_cls(HMM_HMM_PATH, rec['aa'], tokenizer)
        spec_ids, stats = spec_decode_greedy(model, tokenizer, rec['3di'], drafter, K=4, device=device)
        match = spec_ids == ref_ids
        alpha = stats['accepted'] / max(stats['proposed'], 1)
        tag = 'OK  ' if match else 'FAIL'
        print(f"  [{tag}] {rec['uid']:12s} {rec['bucket']:14s} L={rec['length']:4d} {variant_name:12s} a={alpha:.3f}")
        results.append({'uid': rec['uid'], 'bucket': rec['bucket'], 'L': rec['length'], 'variant': variant_name, 'bit_exact': match, 'accept_rate': alpha})
    gc.collect()
    torch.cuda.empty_cache()

import pandas as pd
df = pd.DataFrame(results)
n_ok = int(df.bit_exact.sum())
print(f'\nBit-exact (fp32, K=4): {n_ok}/{len(df)} ({n_ok/len(df)*100:.1f}%)')
df.to_csv(RESULTS_DIR / 'hmm_bit_exact_fp32.csv', index=False)

/tmp/ipykernel_2485/3658456992.py:84: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(FOLDSEEK_DIR)


Verifying bit-exact (fp32) on 6 cases at K=4, both variants



Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


  [OK  ] P39621       in_family      L= 256 naive        a=0.064
  [OK  ] P39621       in_family      L= 256 prefix_aware a=0.058
  [OK  ] P01112       in_family      L= 189 naive        a=0.016
  [OK  ] P01112       in_family      L= 189 prefix_aware a=0.014
  [OK  ] O60762       in_family      L= 260 naive        a=0.098
  [OK  ] O60762       in_family      L= 260 prefix_aware a=0.098
  [OK  ] P04637       out_of_family  L= 393 naive        a=0.017
  [OK  ] P04637       out_of_family  L= 393 prefix_aware a=0.017
  [OK  ] A0A6G0XC32   out_of_family  L= 163 naive        a=0.064
  [OK  ] A0A6G0XC32   out_of_family  L= 163 prefix_aware a=0.059
  [OK  ] P01308       out_of_family  L= 110 naive        a=0.043
  [OK  ] P01308       out_of_family  L= 110 prefix_aware a=0.043

Bit-exact (fp32, K=4): 12/12 (100.0%)


In [8]:
from google.colab import files
files.download(str(RESULTS_DIR / 'hmm_bit_exact_fp32.csv'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Run the benchmark and write summary tables

This final section executes the full experiment.

It first validates the configured drafter variants, then calls `assemble_hmm_cases()` to collect the protein test cases. For each case, it computes the encoder-decoder greedy reference output, benchmarks every drafter variant across all configured `K` values, and stores the generated predictions and metrics.

After the runs finish, it aggregates the raw measurements into several CSV reports:

- per-run raw results,
- per-protein summaries,
- averages split by in-family vs out-of-family buckets,
- a direct bucket comparison table,
- overall averages across all proteins.

The printed tables at the end are a quick inspection view of the same data written under `benchmark_results`.

In [ ]:
unknown_variants = sorted(set(DRAFTER_VARIANTS) - set(DRAFTER_MAP))
if unknown_variants:
    raise ValueError(f'Unknown drafter variants: {unknown_variants}')

hmm_cases = assemble_hmm_cases()
print(f'Selected {len(hmm_cases)} cases')

hmm_all_rows = []
hmm_predictions = {}

for rec in hmm_cases:
    ref_ids = encdec_greedy_reference(model, tokenizer, rec['3di'], device)
    ref_aa = ''.join(tokenizer.decode(ref_ids, skip_special_tokens=True).split())
    hmm_predictions.setdefault(rec['uid'], {})['enc_dec_ref'] = ref_aa

    print(f"--- {rec['uid']} L={rec['length']} {rec['bucket']} ---")
    for variant_name in DRAFTER_VARIANTS:
        drafter_cls = DRAFTER_MAP[variant_name]
        for k in HMM_K_VALUES:
            rows, pred = time_hmm_spec(rec['uid'], rec, drafter_cls, k, ref_ids)
            hmm_all_rows.extend(rows)
            hmm_predictions[rec['uid']][f'{variant_name}_K{k}'] = pred
            row = rows[0]
            print(
                f"  {variant_name:13s} K={k:2d} exact={row['bit_exact']} accept={row['accept_rate']:.3f} "
                f"wall={row['wall_s']:.3f}s tok/s={row['tokens_per_s']:.1f} peak={row['peak_vram_gb']:.2f}GB "
                f"reanchors={row['reanchor_calls']}"
            )

    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

if not hmm_all_rows:
    raise RuntimeError('No HMM benchmark rows were produced.')

hmm_raw = pd.DataFrame(hmm_all_rows)
hmm_raw.to_csv(RESULTS_DIR / 'hmm_raw_runs.csv', index=False)

hmm_summary = (
    hmm_raw.groupby(['protein_id', 'bucket', 'label', 'variant', 'k', 'length'])
           .agg(
               bit_exact=('bit_exact', 'min'),
               accept_rate=('accept_rate', 'median'),
               wall_s=('wall_s', 'median'),
               tokens_per_s=('tokens_per_s', 'median'),
               peak_vram_gb=('peak_vram_gb', 'max'),
               reanchor_calls=('reanchor_calls', 'max'),
           )
           .reset_index()
           .sort_values(['bucket', 'protein_id', 'variant', 'k'])
)
hmm_summary.to_csv(RESULTS_DIR / 'hmm_summary.csv', index=False)

hmm_average_by_bucket = (
    hmm_summary.groupby(['bucket', 'variant', 'k'])
               .agg(
                   proteins=('protein_id', 'nunique'),
                   bit_exact_fraction=('bit_exact', 'mean'),
                   accept_rate=('accept_rate', 'mean'),
                   wall_s=('wall_s', 'mean'),
                   tokens_per_s=('tokens_per_s', 'mean'),
                   peak_vram_gb=('peak_vram_gb', 'mean'),
                   reanchor_calls=('reanchor_calls', 'mean'),
               )
               .reset_index()
               .sort_values(['bucket', 'variant', 'k'])
)
hmm_average_by_bucket.to_csv(RESULTS_DIR / 'hmm_average_by_bucket.csv', index=False)

bucket_comparison = (
    hmm_average_by_bucket
    .pivot(index=['variant', 'k'], columns='bucket', values=[
        'proteins',
        'bit_exact_fraction',
        'accept_rate',
        'wall_s',
        'tokens_per_s',
        'peak_vram_gb',
        'reanchor_calls',
    ])
    .sort_index(axis=1)
)
bucket_comparison.columns = [f'{metric}_{bucket}' for metric, bucket in bucket_comparison.columns]
bucket_comparison = bucket_comparison.reset_index()
bucket_comparison.to_csv(RESULTS_DIR / 'hmm_in_vs_out_comparison.csv', index=False)

hmm_average_overall = (
    hmm_summary.groupby(['variant', 'k'])
               .agg(
                   proteins=('protein_id', 'nunique'),
                   bit_exact_fraction=('bit_exact', 'mean'),
                   accept_rate=('accept_rate', 'mean'),
                   wall_s=('wall_s', 'mean'),
                   tokens_per_s=('tokens_per_s', 'mean'),
                   peak_vram_gb=('peak_vram_gb', 'mean'),
                   reanchor_calls=('reanchor_calls', 'mean'),
               )
               .reset_index()
               .sort_values(['variant', 'k'])
)
hmm_average_overall.to_csv(RESULTS_DIR / 'hmm_average_overall.csv', index=False)

print('=== HMM summary ===')
with pd.option_context('display.float_format', DISPLAY_FLOAT_FORMAT):
    print(hmm_summary.to_string(index=False))

print('\n=== HMM average by bucket ===')
with pd.option_context('display.float_format', DISPLAY_FLOAT_FORMAT):
    print(hmm_average_by_bucket.to_string(index=False))

print('\n=== In-family vs out-of-family comparison ===')
with pd.option_context('display.float_format', DISPLAY_FLOAT_FORMAT):
    print(bucket_comparison.to_string(index=False))

print('\n=== HMM average overall ===')
with pd.option_context('display.float_format', DISPLAY_FLOAT_FORMAT):
    print(hmm_average_overall.to_string(index=False))



print(f'\nSummary written to {RESULTS_DIR / "summary.md"}')

print(f"All runs bit-exact vs enc-dec greedy: {bool(hmm_summary['bit_exact'].all())}")


Selected 20 cases
--- P39621 L=256 in_family ---
  naive         K= 1 exact=False accept=0.196 wall=11.960s tok/s=21.5 peak=5.95GB reanchors=0
  naive         K= 2 exact=False accept=0.115 wall=11.230s tok/s=22.9 peak=5.95GB reanchors=0
  naive         K= 4 exact=False accept=0.064 wall=10.728s tok/s=24.0 peak=5.95GB reanchors=0
  naive         K= 8 exact=False accept=0.033 wall=10.160s tok/s=25.3 peak=5.95GB reanchors=0
  naive         K=16 exact=True accept=0.017 wall=10.978s tok/s=23.4 peak=5.95GB reanchors=0
